# EEG Processing

In this notebook, we assess the quality of the EEG recordings. If possible, we process the EEG data to enhance its quality.

## 📦 Imports

In [1]:
%matplotlib qt

from pathlib import Path 
import os 

import mindmap as mm

import mne

## 🧠 Data

The raw EEG contains the scalp potentials recorded over the entire duration. If onset times are provided, the data can be divided into epochs, which can then be averaged to form evoked potentials. Evoked potentials have the advantage of a strong signal-to-noise ratio, as the noise is averaged out. However, they require onset times, which may not be available in certain situations, such as during resting state recordings.

In [3]:
sub     = "sub-pat10"
mm.io.read_config("../configurations/spike.yml", env=True)

# directories
mm_dir = Path(os.environ["MM_DIR"])

# data
data = mm.io.read_dataset(sub, "eeg_raw")

## Raw

#### <font color=yellow>🌟</font> Visualize Raw

In [4]:
_ = data["eeg_raw"].plot()

In [5]:
_ = data["eeg_raw"].compute_psd().plot()

    Using multitaper spectrum estimation with 7 DPSS windows


Plotting power spectral density (dB=True).


#### Process Raw

⚙️ <font color=green>**Interpolate bad channels**</font>: After observing the time series and Fourier transform, we can identify any electrodes that did not record properly. In such cases, we interpolate the faulty channels to prevent them from affecting the rest of the analysis.

In [8]:
data["eeg_raw"].info["bads"].append("7")
data["eeg_raw"].interpolate_bads()

Setting channel interpolation method to {'eeg': 'spline'}.
Interpolating bad channels.
    Automatic origin fit: head of radius 88.7 mm
Computing interpolation matrix from 203 sensor positions
Interpolating 1 sensors


Condition,
Data kind,average
Timepoints,250 samples
Channels,204 channels
Number of averaged epochs,1
Time range (secs),0.0 – 0.996
Baseline (secs),off


⚙️ <font color=green>**Set reference**</font>: We can also determine whether the reference electrode is too close to the neural sources. An ideal reference signal captures none of the brain-specific fluctuations in electric potential while picking up all of the environmental noise and interference detected by the measurement electrodes. In practice, this means that the reference electrode is often placed on the subject's body, close to the head but as far away from the neural sources as possible. If the reference electrode also records neural sources, we can set a "virtual reference," which is the average of all channels.

In [6]:
data["eeg_raw"].set_eeg_reference(ref_channels="average")
_ = data["eeg_raw"].plot()

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


## Epoched

## Evoked

In [7]:
if os.environ["NAME"] == "spike":
    evoked = data["eeg_raw"].copy()

elif os.environ["NAME"] == "intra":
    raw = data["eeg_raw"].set_eeg_reference(["VREF"])
    evoked = raw.average()
    evoked = mne.set_eeg_reference(evoked, "average")

_ = evoked.plot()

#### Process Evoked

⚙️ <font color=green>**Baseline correction**</font>: To accurately compare EEG channels, they must have the same baseline.

In [9]:
baseline = [0, 0.3]
evoked.apply_baseline(baseline)

_ = evoked.plot()

Applying baseline correction (mode: mean)


⚙️ <font color=green>**Notch Filtering**</font>: A prevalent source of noise in recordings from electrical devices is the 50 Hz electrical interference. To mitigate this type of noise, it is advisable to apply a notch filter, which is a band-stop filter specifically designed to attenuate frequencies around 50 Hz.

In [ ]:
evoked.filter(
    l_freq = 63,
    h_freq = 37,
    picks  = "all",
    method = "fir",
)

_ = evoked.plot()

## Visualize Topomaps

It is important to check both the EEG time series and the topological maps they generate. A topological map should be smooth; if it is not, the electrodes responsible for the irregularities are likely faulty.

In [6]:
# time points 
times = [0.4, 0.45, 0.5, 0.55, 0.6]

_ = evoked.plot_topomap(
    times = times,
    outlines = None,
    extrapolate = "local",
)


## 💾 Save processed EEG

In [7]:
# save source space
folder = mm_dir/sub/"eeg"

if folder.exists() == False:
        folder.mkdir(parents=True)

print(f"Save EEG for subject: {sub}")
evoked.save(folder/f"{sub}_clean-ave.fif", overwrite=True)

Save EEG for subject: sub-pat10
Overwriting existing file.
